# 03 · Creación del pipeline de features: Predict Future Sales

**Objetivo:** implementar como un `Pipeline` de scikit-learn el proceso de ingeniería de características definido en el notebook 02, y guardarlo en `models/pipeline_fe.pkl`.

El pipeline recibe filas **crudas** `(date_block_num, shop_id, item_id)`, como las de `test.csv` con `date_block_num = 34`, y devuelve 35 columnas numéricas listas para el modelo. Usa dos operadores propios, definidos en `future_sales/features.py`:

- **`Mapper`**: agrega columnas mapeando otra columna con un diccionario (producto → categoría, tienda → ciudad, mes → atributos de calendario). Es el operador visto en clase.
- **`GeneradorLags`**: agrega las ventas de meses anteriores a partir del historial mensual, **solo con meses anteriores** al de la fila.

Los demás pasos son transformadores de `feature-engine` y `scikit-learn`. La configuración (lags, listas de variables) se lee de `pipeline_config.yaml`, generado por el notebook 02.

## Configuración

Librerías, configuración del notebook 02 y semilla.

In [1]:
import time

from IPython.display import display
from feature_engine.encoding import MeanEncoder, OneHotEncoder, RareLabelEncoder
from feature_engine.imputation import (
    AddMissingIndicator,
    ArbitraryNumberImputer,
    CategoricalImputer,
    MeanMedianImputer,
)
from feature_engine.outliers import ArbitraryOutlierCapper, Winsorizer
from feature_engine.selection import DropFeatures
from feature_engine.transformation import LogCpTransformer
from feature_engine.wrappers import SklearnTransformerWrapper
import joblib
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import yaml

from future_sales import dataset as ds
from future_sales.config import MODELS_DIR, PROJ_ROOT
from future_sales.features import (
    COLUMNAS_HISTORIAL,
    GeneradorLags,
    Mapper,
    construir_mapeos,
)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)
pd.set_option("display.float_format", "{:,.3f}".format)

SEMILLA = 42
with open(PROJ_ROOT / "pipeline_config.yaml", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)
LAGS = CFG["LAGS"]
print("Lags:", LAGS)

2026-11-20 08:59:32.258 | INFO     | future_sales.config:<module>:11 - PROJ_ROOT path is: C:\Users\rfarfan\OneDrive - Universidad Galileo\2026T4\Product Development\Proyecto\future-sales


Lags: {'par': [1, 2, 3, 6, 12], 'item': [1, 2], 'tienda': [1], 'categoria': [1], 'tienda_categoria': [1, 2]}


## 1. Datos de entrada

1. Ventas limpias y agregadas por mes (`future_sales.dataset`), igual que en el notebook 02.
2. Malla de los meses 12 a 33, con el target recortado a [0, 20].
3. División **temporal**: entrenamiento con los meses 12-26 y validación con los meses 27-33.

`X` tiene solo las tres columnas que llegan en producción; todo lo demás lo genera el pipeline.

In [2]:
inicio = time.time()
catalogos = ds.cargar_catalogos()
mensual = ds.ventas_mensuales(ds.limpiar_ventas(ds.cargar_ventas()),
                              catalogos["items"])
malla = ds.construir_malla(mensual, range(12, 34))

ENTRADA = ["date_block_num", "shop_id", "item_id"]
desde, hasta = CFG["MESES_ENTRENAMIENTO"]
es_entrenamiento = malla["date_block_num"].between(desde, hasta)
X_ent = malla.loc[es_entrenamiento, ENTRADA]
y_ent = malla.loc[es_entrenamiento, CFG["TARGET"]]
X_val = malla.loc[~es_entrenamiento, ENTRADA]
y_val = malla.loc[~es_entrenamiento, CFG["TARGET"]]
print(f"Entrenamiento (meses {desde}-{hasta}): {len(X_ent):,} filas")
print(f"Validación (meses {CFG['MESES_VALIDACION']}): {len(X_val):,} filas")
print(f"({time.time() - inicio:.1f} s)")

# Muestra para mostrar el efecto de cada paso.
X_muestra = X_ent.sample(200_000, random_state=SEMILLA)
y_muestra = y_ent.loc[X_muestra.index]
X_muestra.head()

Entrenamiento (meses 12-26): 4,810,730 filas
Validación (meses [27, 33]): 1,614,364 filas
(2.1 s)


,date_block_num,shop_id,item_id
2314661,18,56,18072
1865206,17,39,1329
1807521,17,26,9929
343878,12,59,12266
2483039,19,30,3708


## 2. Paso 1: mapeos con `Mapper`

Agrega las variables exógenas de la tienda y del producto, y los atributos de calendario del mes. Los diccionarios se construyen con `construir_mapeos` a partir de los catálogos traducidos (`data/external`) y del calendario.

In [3]:
mapeos = construir_mapeos(catalogos, ds.calendario())
pasos_mapeo = [("mapear", Mapper(mapeos))]

for nueva, (origen, diccionario) in mapeos.items():
    print(f"{nueva:<20} ← {origen:<16} ({len(diccionario):,} valores)")
X1 = Pipeline(pasos_mapeo).fit_transform(X_muestra)
X1.head()

item_category_id     ← item_id          (22,170 valores)
categoria_principal  ← item_category_id (84 valores)
es_digital           ← item_category_id (84 valores)
ciudad               ← shop_id          (60 valores)
tipo_tienda          ← shop_id          (60 valores)
mes                  ← date_block_num   (35 valores)
dias_mes             ← date_block_num   (35 valores)
fines_semana         ← date_block_num   (35 valores)


,date_block_num,shop_id,item_id,item_category_id,categoria_principal,es_digital,ciudad,tipo_tienda,mes,dias_mes,fines_semana
2314661,18,56,18072,62,Regalos,0,Chéjov,Centro comercial y de ocio,7,31,8
1865206,17,39,1329,55,Música,0,Rostov del Don,Centro comercial y de ocio,6,30,9
1807521,17,26,9929,40,Cine,0,Moscú,Centro comercial,6,30,9
343878,12,59,12266,62,Regalos,0,Yaroslavl,Centro comercial,1,31,8
2483039,19,30,3708,19,Juegos,0,Moscú,Centro comercial,8,31,10


## 3. Paso 2: lags con `GeneradorLags`

El historial es la tabla de ventas mensuales de los meses 0 a 33. Para una fila del mes `m`, cada lag `k` toma el valor del mes `m - k`. El operador guarda el historial dentro del pipeline, por eso el `.pkl` puede predecir el mes 34 a partir de `test.csv` sin más datos.

In [4]:
historial = mensual[COLUMNAS_HISTORIAL]
pasos_lags = [("generar_lags", GeneradorLags(
    historial=historial,
    lags_par=LAGS["par"],
    lags_item=LAGS["item"],
    lags_tienda=LAGS["tienda"],
    lags_categoria=LAGS["categoria"],
    lags_tienda_categoria=LAGS["tienda_categoria"],
))]
X2 = Pipeline(pasos_lags).fit_transform(X1)
print(f"Historial: {len(historial):,} filas (meses "
      f"{historial['date_block_num'].min()} a "
      f"{historial['date_block_num'].max()})")
X2.iloc[:, len(X1.columns):].describe().T

Historial: 1,609,123 filas (meses 0 a 33)


,count,mean,std,min,25%,50%,75%,max
ventas_par_lag_1,"200,000.000",0.348,4.651,-1.000,0.000,0.000,0.000,"1,074.000"
ventas_par_lag_2,"200,000.000",0.347,3.531,-1.000,0.000,0.000,0.000,795.000
ventas_par_lag_3,"200,000.000",0.341,3.663,-1.000,0.000,0.000,0.000,764.000
ventas_par_lag_6,"200,000.000",0.316,3.406,-2.000,0.000,0.000,0.000,759.000
ventas_par_lag_12,"200,000.000",0.268,2.620,-2.000,0.000,0.000,0.000,634.000
ventas_item_lag_1,"200,000.000",16.642,115.404,-18.000,1.000,4.000,12.000,"12,557.000"
ventas_item_lag_2,"200,000.000",16.890,106.437,-22.000,1.000,4.000,12.000,"12,185.000"
ventas_tienda_lag_1,"200,000.000","2,323.767","1,968.110",0.000,"1,222.000","1,738.000","2,519.000","16,318.000"
ventas_categoria_lag_1,"200,000.000","7,163.347","6,822.805",0.000,"1,018.000","5,647.000","11,533.000","28,598.000"
ventas_tienda_categoria_lag_1,"200,000.000",146.953,259.303,-1.000,7.000,66.000,179.000,"4,824.000"


### 3.1 Prueba de fuga de información

Si los lags usan solo meses anteriores, las features de una fila del mes `m` deben ser **idénticas** aunque se borre del historial todo lo ocurrido desde el mes `m` en adelante. Se prueba con filas de validación del mes 30: se generan con el historial completo y con un historial que termina en el mes 29.

In [5]:
filas_mes_30 = Pipeline(pasos_mapeo).fit_transform(
    X_val[X_val["date_block_num"] == 30].sample(20_000,
                                                 random_state=SEMILLA))
con_todo = GeneradorLags(historial, **{
    f"lags_{nivel}": lags for nivel, lags in LAGS.items()
}).fit(filas_mes_30).transform(filas_mes_30)
hasta_29 = GeneradorLags(historial[historial["date_block_num"] < 30], **{
    f"lags_{nivel}": lags for nivel, lags in LAGS.items()
}).fit(filas_mes_30).transform(filas_mes_30)
pd.testing.assert_frame_equal(con_todo, hasta_29)
print("Sin fuga: las features del mes 30 no cambian al quitar los meses "
      "30 a 33 del historial.")

Sin fuga: las features del mes 30 no cambian al quitar los meses 30 a 33 del historial.


## 4. Paso 3: imputación

- **Numéricas con bandera:** `AddMissingIndicator` crea las banderas de producto nuevo y par nuevo; después, `ArbitraryNumberImputer(0)` rellena la antigüedad.
- **Numéricas:** mediana para el precio previo, los lags y los atributos de calendario.
- **Categóricas:** `"Desconocido"` para ciudad, tipo de tienda y categoría principal; `-1` para el código de categoría.

In [6]:
pasos_imputacion = [
    ("indicador_faltantes", AddMissingIndicator(
        variables=CFG["IMPUTAR_CERO"])),
    ("imputar_cero", ArbitraryNumberImputer(
        arbitrary_number=0, variables=CFG["IMPUTAR_CERO"])),
    ("imputar_mediana", MeanMedianImputer(
        imputation_method="median", variables=CFG["IMPUTAR_MEDIANA"])),
    ("imputar_categoricas", CategoricalImputer(
        imputation_method="missing", fill_value="Desconocido",
        variables=CFG["IMPUTAR_CATEGORICAS"])),
    ("imputar_codigos", ArbitraryNumberImputer(
        arbitrary_number=-1, variables=CFG["IMPUTAR_CODIGOS"])),
]
print("Faltantes antes:", X2.isna().sum()[X2.isna().sum() > 0].to_dict())
X3 = Pipeline(pasos_imputacion).fit_transform(X2)
print("Faltantes después:", int(X3.isna().sum().sum()))
print("Banderas:", [c for c in X3.columns if c.endswith("_na")])

Faltantes antes: {'precio_item_previo': 9958, 'meses_desde_lanzamiento_item': 9958, 'meses_desde_primera_venta_par': 95719}
Faltantes después: 0
Banderas: ['meses_desde_lanzamiento_item_na', 'meses_desde_primera_venta_par_na']


## 5. Paso 4: codificación

- `tipo_tienda` y `categoria_principal`: `RareLabelEncoder` (categorías con menos del 1 % → "Rare") y `OneHotEncoder`.
- `shop_id`, `item_category_id` y `ciudad`: `MeanEncoder` con suavizado automático. Las categorías que no se vieron en el ajuste reciben la media global (`unseen="encode"`).

In [7]:
pasos_codificacion = [
    ("agrupar_raras", RareLabelEncoder(
        tol=0.01, n_categories=4, variables=CFG["CODIFICAR_ONE_HOT"])),
    ("one_hot", OneHotEncoder(variables=CFG["CODIFICAR_ONE_HOT"])),
    ("media_target", MeanEncoder(
        variables=CFG["CODIFICAR_MEDIA_TARGET"], smoothing="auto",
        unseen="encode", ignore_format=True)),
]
X4 = Pipeline(pasos_codificacion).fit_transform(X3, y_muestra)
print("Columnas one-hot:",
      [c for c in X4.columns if c not in X3.columns])
X4[CFG["CODIFICAR_MEDIA_TARGET"]].describe().T

Columnas one-hot: ['tipo_tienda_Centro comercial y de ocio', 'tipo_tienda_Centro comercial', 'tipo_tienda_Tienda de calle', 'tipo_tienda_En línea', 'tipo_tienda_Rare', 'categoria_principal_Regalos', 'categoria_principal_Música', 'categoria_principal_Cine', 'categoria_principal_Juegos', 'categoria_principal_Rare', 'categoria_principal_Programas', 'categoria_principal_Accesorios', 'categoria_principal_Libros']


,count,mean,std,min,25%,50%,75%,max
shop_id,"200,000.000",0.293,0.216,0.056,0.177,0.226,0.305,1.168
item_category_id,"200,000.000",0.290,0.315,0.000,0.166,0.226,0.243,7.627
ciudad,"200,000.000",0.293,0.151,0.082,0.184,0.215,0.495,0.727


## 6. Paso 5: outliers

- Todos los lags con piso 0 (devoluciones netas) y los lags del par con tope 20, igual que el target (`ArbitraryOutlierCapper`).
- Lags agregados y precio: `Winsorizer` por cuantiles, cola derecha al percentil 99.5.

In [8]:
lags_todos = [f"ventas_{nivel}_lag_{k}"
              for nivel, ks in LAGS.items() for k in ks]
pasos_outliers = [
    ("recortar_lags", ArbitraryOutlierCapper(
        max_capping_dict={c: 20 for c in CFG["RECORTAR_LAGS_PAR"]},
        min_capping_dict={c: 0 for c in lags_todos})),
    ("winsorizar", Winsorizer(
        capping_method="quantiles", tail="right", fold=0.005,
        variables=CFG["WINSORIZAR"])),
]
X5 = Pipeline(pasos_outliers).fit_transform(X4)
limites = pd.DataFrame({
    "max_antes": X4[CFG["WINSORIZAR"]].max(),
    "tope_aprendido": pd.Series(
        Pipeline(pasos_outliers).fit(X4).named_steps["winsorizar"]
        .right_tail_caps_),
    "max_despues": X5[CFG["WINSORIZAR"]].max(),
})
print("Lags considerados:", lags_todos)
limites

Lags considerados: ['ventas_par_lag_1', 'ventas_par_lag_2', 'ventas_par_lag_3', 'ventas_par_lag_6', 'ventas_par_lag_12', 'ventas_item_lag_1', 'ventas_item_lag_2', 'ventas_tienda_lag_1', 'ventas_categoria_lag_1', 'ventas_tienda_categoria_lag_1', 'ventas_tienda_categoria_lag_2']


,max_antes,tope_aprendido,max_despues
ventas_item_lag_1,"12,557.000",296.005,296.005
ventas_item_lag_2,"12,185.000",313.000,313.000
ventas_tienda_lag_1,"16,318.000","12,102.000","12,102.000"
ventas_categoria_lag_1,"28,598.000","28,598.000","28,598.000"
ventas_tienda_categoria_lag_1,"4,824.000","1,787.000","1,787.000"
ventas_tienda_categoria_lag_2,"4,824.000","1,835.000","1,835.000"
precio_item_previo,"36,990.000","8,649.000","8,649.000"


## 7. Paso 6: transformación

`log(1 + x)` en las variables en las que el notebook 02 mostró que el logaritmo reduce el sesgo.

In [9]:
pasos_transformacion = [
    ("log1p", LogCpTransformer(variables=CFG["TRANSFORMAR_LOG"], C=1)),
]
X6 = Pipeline(pasos_transformacion).fit_transform(X5)
pd.DataFrame({"sesgo_antes": X5[CFG["TRANSFORMAR_LOG"]].skew(),
              "sesgo_despues": X6[CFG["TRANSFORMAR_LOG"]].skew()})

,sesgo_antes,sesgo_despues
ventas_par_lag_1,9.158,3.392
ventas_par_lag_2,9.151,3.445
ventas_par_lag_3,9.161,3.497
ventas_par_lag_6,9.646,3.663
ventas_par_lag_12,10.036,4.093
ventas_item_lag_1,5.339,0.562
ventas_item_lag_2,5.486,0.573
ventas_categoria_lag_1,0.928,-0.836
ventas_tienda_categoria_lag_1,3.673,-0.488
ventas_tienda_categoria_lag_2,3.658,-0.466


## 8. Paso 7: eliminación y escalado

Se eliminan las columnas que no son features (`date_block_num`, `item_id`, `dias_mes`) y se escalan todas las restantes con `StandardScaler`.

In [10]:
pasos_escalado = [
    ("eliminar", DropFeatures(features_to_drop=CFG["ELIMINAR"])),
    ("escalar", SklearnTransformerWrapper(StandardScaler())),
]
X7 = Pipeline(pasos_escalado).fit_transform(X6)
print(f"Columnas finales: {X7.shape[1]}")
X7.agg(["mean", "std"]).T.round(3).head(10)

Columnas finales: 35


,mean,std
shop_id,-0.000,1.000
item_category_id,0.000,1.000
es_digital,-0.000,1.000
ciudad,0.000,1.000
mes,0.000,1.000
fines_semana,-0.000,1.000
ventas_par_lag_1,0.000,1.000
ventas_par_lag_2,0.000,1.000
ventas_par_lag_3,0.000,1.000
ventas_par_lag_6,-0.000,1.000


## 9. Pipeline final

El pipeline completo es la concatenación de los pasos anteriores. Se ajusta con **todo** el entrenamiento (meses 12-26) y se transforma la validación (meses 27-33). Los imputadores, los codificadores, los topes y el escalado aprenden solo del entrenamiento.

In [11]:
pipeline_fe = Pipeline(
    pasos_mapeo
    + pasos_lags
    + pasos_imputacion
    + pasos_codificacion
    + pasos_outliers
    + pasos_transformacion
    + pasos_escalado
)
inicio = time.time()
Xt_ent = pipeline_fe.fit_transform(X_ent, y_ent)
Xt_val = pipeline_fe.transform(X_val)
print(f"Ajuste y transformación en {time.time() - inicio:.0f} s")
print(f"Entrenamiento: {Xt_ent.shape} | validación: {Xt_val.shape}")
print("Faltantes:", int(Xt_ent.isna().sum().sum()),
      int(Xt_val.isna().sum().sum()))
assert list(Xt_ent.columns) == list(Xt_val.columns)
print("Pasos:", [nombre for nombre, _ in pipeline_fe.steps])
print("Columnas:", list(Xt_ent.columns))

Ajuste y transformación en 31 s
Entrenamiento: (4810730, 35) | validación: (1614364, 35)
Faltantes: 0 0
Pasos: ['mapear', 'generar_lags', 'indicador_faltantes', 'imputar_cero', 'imputar_mediana', 'imputar_categoricas', 'imputar_codigos', 'agrupar_raras', 'one_hot', 'media_target', 'recortar_lags', 'winsorizar', 'log1p', 'eliminar', 'escalar']
Columnas: ['shop_id', 'item_category_id', 'es_digital', 'ciudad', 'mes', 'fines_semana', 'ventas_par_lag_1', 'ventas_par_lag_2', 'ventas_par_lag_3', 'ventas_par_lag_6', 'ventas_par_lag_12', 'ventas_item_lag_1', 'ventas_item_lag_2', 'ventas_tienda_lag_1', 'ventas_categoria_lag_1', 'ventas_tienda_categoria_lag_1', 'ventas_tienda_categoria_lag_2', 'precio_item_previo', 'meses_desde_lanzamiento_item', 'meses_desde_primera_venta_par', 'meses_desde_lanzamiento_item_na', 'meses_desde_primera_venta_par_na', 'tipo_tienda_Centro comercial', 'tipo_tienda_Centro comercial y de ocio', 'tipo_tienda_Tienda de calle', 'tipo_tienda_En línea', 'tipo_tienda_Rare', '

### 9.1 Transformación de test (mes 34)

El mismo pipeline transforma las filas crudas de `test.csv`, agregándoles `date_block_num = 34`. Los lags salen de los meses 33, 32, …, 22 del historial.

In [12]:
test = pd.read_csv(ds.RAW_DATA_DIR / "test.csv", index_col="ID")
X_test = test.assign(date_block_num=ds.MES_TEST,
                     shop_id=ds.unificar_tiendas(test["shop_id"]))[ENTRADA]
Xt_test = pipeline_fe.transform(X_test)
print(f"Test: {Xt_test.shape} | faltantes: {int(Xt_test.isna().sum().sum())}")
lags_test = pipeline_fe[:2].transform(X_test)
lags_test[["ventas_par_lag_1", "ventas_item_lag_1",
           "meses_desde_lanzamiento_item"]].describe().T

Test: (214200, 35) | faltantes: 0


,count,mean,std,min,25%,50%,75%,max
ventas_par_lag_1,"214,200.000",0.283,2.675,-1.000,0.000,0.000,0.000,500.000
ventas_item_lag_1,"214,200.000",12.864,68.548,0.000,1.000,4.000,11.000,"4,078.000"
meses_desde_lanzamiento_item,"198,954.000",17.417,11.663,1.000,7.000,15.000,30.000,34.000


## 10. Guardar el pipeline

Se guarda **ajustado** con el entrenamiento en `models/pipeline_fe.pkl` (joblib, comprimido). El notebook 04 lo clona sin ajustar (`sklearn.base.clone`) para cada modelo del torneo, así que la configuración es la misma y el ajuste se repite con los datos de cada caso.

In [13]:
ruta_fe = MODELS_DIR / "pipeline_fe.pkl"
joblib.dump(pipeline_fe, ruta_fe, compress=3)
print(f"{ruta_fe.name}: {ruta_fe.stat().st_size / 1e6:.1f} MB")

recargado = joblib.load(ruta_fe)
pd.testing.assert_frame_equal(recargado.transform(X_val.head(1_000)),
                              Xt_val.head(1_000))
print("El pipeline recargado produce las mismas features.")

pipeline_fe.pkl: 21.9 MB
El pipeline recargado produce las mismas features.
